# UFC Outcomes & Fighter Trajectories: Exploring the Dynamics of Mixed Martial Arts through Data Analysis and Visualization

I have been a dedicated fan of the UFC for the last 10 years, but my fascination peaked during the COVID-19 pandemic when the sport was the only LIVE sport on TV. What interests me most about MMA is the complex matchups of two fighter, with sometimes very distinct skill sets, colliding inside the Octagon across weight-classes.  Do great grapplers really beat great strikers, does weight class matter?? There are many very interesting insights that want to explore with this data

Today, I will be diving into a comprehensive UFC bout dataset from kaggle to find these insights in the matchups.  In my view, a prize fighters performance is entirely dependent on stylist advantages, physical measures, and career legnth...but lets see what the data tells us!

---

### The Core Objectives of This Exploration:
**Pre-Fight Matchups & Style Collisions:** I want to analyze how fighters match up against one another based purely on the data known before they step into the cage. By filtering metrics like striking accuracy, takedown attempts, and reach advantages, I hope to discover what combinations of traits create a winning formula.

**Predictive Correlations:** I will map pre-fight career statistics against the actual fight results to identify which specific metrics hold the highest statistical correlation to a victory.

**The Aging Curve:** Does age matter and at what point do fighters start going down hill?  I want to visualize fighter trajectories over time to see if clear trends emerge as fighters age. Does a drop in striking volume or an increase in damage absorbed signal the inevitable cliff of an aging athlete?


---

### Visualization Narrative
The following outlines the 4 charts (from upper left to lower right) that are in the dashboard and their visualizations to meet the objectiv

- **Correlation Matrix (What correlates with a win and what does not?)**: This chart displays the Pearson correlation coefficients betwwen the top 10 most impactful fight/fighter statistics and the target variable (Red_Is_Winner). It uses a horizontal layout with a color scale (red for positive correlation, blue for negative) and sorts the bars by magnitude to immediately highlight which metrics have the strongest relationship to winning a fight.
- **Scatter Plot (Physical Attributes):** This visualization plots the relationship between two continuous variables: Fighter Height and Fighter Reach. It utilizes color-coding to segment the data points by weight class and incorporates jitter to prevent overplotting, allowing for the clear identification of physical clusters and outliers of unique fighters.
- **Area Chart (Age Difference Win Distribution):** This chart plots the distribution of total wins across categorical, relative age brackets (e.g., "2-4 Yrs Younger", "Even - 1 Yr"). By mapping the counts to the Y-axis and filling the area beneath the connecting lines, it creates a distinct distribution curve, separated by color to denote gender.
- **Line Chart (Performance Aging Curve):** This temporal chart tracks specific performance metrics (such as Average Takedowns Landed) against a fighter's absolute age. Multiple lines are rendered simultaneously, each representing a different weight class, to track how physical output naturally trends or decays as fighters progress through their careers.

On the left side of the Dashboard are the different preset selectors that can be used to filter the data. All the charts can be filted by Gender, Weight Class and Fighter stance.  The Aging curve is the onge chart taht can be filtered by fight_metric to see the change in mertic over time.

## Visualization Technique

## Visualization Library

**Plotly** is a powerful open-source graphing library built on top of plotly.js, which means it natively renders interactive, web-ready visualizations directly in browser-based environments like Jupyter Notebooks. It offers a massive range of charts from basic scatter, line, and bar plots to more complex heatmaps, 3D scatter plots, and geographical maps making it great for analyzing sports data. Because it is JavaScript-native, features like zooming, panning, and instant hover-tooltips work out of the box without requiring a separate web server. This makes it an ideal fit for building a dynamic dashboard where we can seamlessly toggle between weight classes or isolate specific fighter metrics without a clunky lag.

Link: https://plotly.com/python/


!pip isntall plotly

**Panel** is an open-source Python library designed to streamline the development of robust tools, dashboards, and complex applications entirely within Python.  I had never used Panel before this course and was interested to see how I could use plotly for the graphs and Panel for the dashboard capability.  I found it simple and lightweight to use for dashboard architecture and interactive selections and toggles to update my underlyind graphs.

Link: https://panel.holoviz.org/


!pip install panel


# Setup and Dependencies

Start by installing the libraries if they are not already in your environment


In [1]:
!pip install panel pandas numpy plotly plotly_express kagglehub

Once Dependencies are in your environment, import them

In [2]:
import kagglehub
import pandas as pd
import panel as pn
import numpy as np
import seaborn as sns
import plotly.express as px

## Data: Collecting and Cleaning

I will be using UFC data from https://www.kaggle.com/datasets/mdabbert/ultimate-ufc-dataset/data

NOTE: make sure you pip install and import kagglehub before proceeding


In [3]:
# get the path of the latest dataset
path = kagglehub.dataset_download("mdabbert/ultimate-ufc-dataset")
print("Path to dataset files:", path)

# create a pandas dataframe from the CSV file based on the path of the files
df = pd.read_csv(path + "/ufc-master.csv")

#inpect your data
df.head()

Path to dataset files: /Users/michaelsnow/.cache/kagglehub/datasets/mdabbert/ultimate-ufc-dataset/versions/181


,R_fighter,B_fighter,R_odds,B_odds,R_ev,B_ev,date,location,country,Winner,...,finish_details,finish_round,finish_round_time,total_fight_time_secs,r_dec_odds,b_dec_odds,r_sub_odds,b_sub_odds,r_ko_odds,b_ko_odds
0,Israel Adesanya,Joe Pyfer,-130.0,102.0,76.9231,102.0000,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Punches,2.0,4:18,558.0,163.0,900.0,2500.0,400.0,300.0,250.0
1,Alexa Grasso,Maycee Barber,124.0,-158.0,124.0000,63.2911,2026-03-28,"Seattle, Washington, USA",USA,Red,...,Punch,1.0,2:42,162.0,175.0,105.0,1400.0,800.0,2500.0,500.0
2,Michael Chiesa,Niko Price,-901.0,550.0,11.0988,550.0000,2026-03-28,"Seattle, Washington, USA",USA,Red,...,Rear Naked Choke,1.0,1:03,63.0,225.0,900.0,-150.0,1600.0,600.0,1000.0
3,Julian Erosa,Lerryan Douglas,235.0,-320.0,235.0000,31.2500,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Punches,1.0,3:33,213.0,600.0,500.0,600.0,2000.0,700.0,-150.0
4,Mansur Abdul-Malik,Yousri Belgaroui,-158.0,124.0,63.2911,124.0000,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Knee,3.0,3:39,819.0,350.0,240.0,800.0,1800.0,240.0,250.0


We will now go into exploring the data, cleaning it and concentrating on the columns we care about for our EDA objectives stated above.

In [4]:
#understand what columns have null values
df.isnull().sum()

R_fighter        0
B_fighter        0
R_odds         240
B_odds         239
R_ev           240
              ... 
b_dec_odds    1145
r_sub_odds    1364
b_sub_odds    1388
r_ko_odds     1362
b_ko_odds     1389
Length: 118, dtype: int64

In [ ]:
# we will drop the following columns from the dataset as they are not relevant for our analysis.  Some are Fight results, some do not have enough info,
#  and some are betting odds that are already a form of predictive model
columns_to_drop = [
    'empty_arena',
    'B_match_weightclass_rank',
    'R_match_weightclass_rank',
    "R_Women's Flyweight_rank",
    "R_Women's Featherweight_rank",
    "R_Women's Strawweight_rank",
    "R_Women's Bantamweight_rank",
    'R_Heavyweight_rank',
    'R_Light Heavyweight_rank',
    'R_Middleweight_rank',
    'R_Welterweight_rank',
    'R_Lightweight_rank',
    'R_Featherweight_rank',
    'R_Bantamweight_rank',
    'R_Flyweight_rank',
    'R_Pound-for-Pound_rank',
    "B_Women's Flyweight_rank",
    "B_Women's Featherweight_rank",
    "B_Women's Strawweight_rank",
    "B_Women's Bantamweight_rank",
    'B_Heavyweight_rank',
    'B_Light Heavyweight_rank',
    'B_Middleweight_rank',
    'B_Welterweight_rank',
    'B_Lightweight_rank',
    'B_Featherweight_rank',
    'B_Bantamweight_rank',
    'B_Flyweight_rank',
    'B_Pound-for-Pound_rank',
    'better_rank',
    'finish',
    'finish_details',
    'finish_round',
    'finish_round_time',
    'total_fight_time_secs',
    'r_dec_odds',
    'b_dec_odds',
    'r_sub_odds',
    'b_sub_odds',
    'r_ko_odds',
    'b_ko_odds',
    'R_odds',
    'B_odds',
    'R_ev',
    'B_ev'
 ]

# we want to keep these columns for our analysis but remove them from the dataset for numerical analysis
columns_remove_for_numerical_analysis = [
    'date',
    'location',
    'country',
    'Winner',
    'B_fighter',
    'R_fighter',
    'title_bout',
    'weight_class',
    'gender',
    'B_Stance',
    'R_Stance'
]

# Create a binary column for stance conflict.  Do the fighters' stances match up or not?
df['mirror_stance_matchup'] = ((df['R_Stance'] == 'Orthodox') & (df['B_Stance'] == 'Southpaw')) | \
                                    ((df['R_Stance'] == 'Southpaw') & (df['B_Stance'] == 'Orthodox'))

df['mirror_stance_matchup'] = df['mirror_stance_matchup'].astype(int)

#now i need to encode categorical variables into numerical values for analysis. I will use one-hot encoding for this purpose.
encode_cols = [
    'gender',
    'weight_class',
    'B_Stance',
    'R_Stance'
]

encoded_col_names = [f'{col}_encoded' for col in encode_cols]

#create a copy of encoded_cols so we can keep the original list intact for later use
for col in encode_cols:
    df[f'{col}_encoded'] = df[col]

#encoding categorical variables using one-hot encoding
df = pd.get_dummies(df, columns=encoded_col_names, drop_first=False)

try:
    subset_df = df.drop(columns=columns_to_drop)
except KeyError as e:
    print(f"Error occurred while dropping columns: {e}")

## Chart Creation

### Area Chart (Age Difference Win Distribution)

In [ ]:
# We are going to see what age differences have the most impact on the outcome of a fight. We will look at the difference across weight class and gender

#create the variable to filter results by age difference
gender_toggle = pn.widgets.RadioButtonGroup(
    name='Gender',
    options=['Both','Male','Female'],
    button_type='primary'
)

#create a variable Selector to filter results by weightclass
weigh_class_dropdown = pn.widgets.Select(
    name='Weight Class',
    options=['All'] + list(subset_df['weight_class'].dropna().unique()),
    value='All'
)

#create a variable Selector to filter results by fighter stance
stance_dropdown = pn.widgets.Select(
    name='Fighter Stance',
    options=['All']+ pd.concat([subset_df['R_Stance'],subset_df['B_Stance']]).str.strip().dropna().unique().tolist(),
    value= 'All'
)

#creaet a function for our Area chart that is dynamic
def plot_age_difference_vs_winner(gender, weight_category, stance):
    #create a copy of the subset_df so we do not override it
    filtered_df = subset_df.copy()

    #create an age difference column
    filtered_df['winner_age_dif'] = np.where(
        filtered_df['Winner']=='Red',
        filtered_df['R_age'] - filtered_df['B_age'],
        filtered_df['B_age'] - filtered_df['R_age']
    )

    #create a winner stance column
    filtered_df['Winner_Stance'] = np.where(
        filtered_df['Winner'] == 'Red',
        filtered_df['R_Stance'],
        filtered_df['B_Stance']
    )

    #create age buckets
    bins = [-100,-9.5,-5,-1.5,1.5,5,9.5,100]
    labels = [
        '10+ Yrs Younger',
        '5-9 Yrs Younger',
        '2-4 Yrs Younger',
        'Even +/- 1Yr',
        '2-4 Yrs Older',
        '5-9 Yrs Older',
        '10+ Yrs Older',
    ]

    filtered_df['Age_Bracket'] = pd.cut(
        filtered_df['winner_age_dif'],
        bins=bins,
        labels=labels
    )

    #apply winner stance filter
    if stance != 'All':
        filtered_df = filtered_df[filtered_df['Winner_Stance'].str.strip() == stance]

    #apply gender filter
    if gender == 'Female':
        filtered_df = filtered_df[filtered_df['gender']=='FEMALE']
    if gender =='Male':
        filtered_df = filtered_df[filtered_df['gender']=='MALE']

    #apply Weight Class filter
    if weight_category != 'All':
        filtered_df = filtered_df[filtered_df['weight_class']== weight_category]

    #count the wins for both Red and Blue in each bucket
    win_counts = (
        filtered_df.groupby(['Age_Bracket','gender'],observed=False).size().reset_index(name='Count')
    )

    #create the plotly area chart
    fig = px.area(win_counts,
                     x='Age_Bracket', 
                     y='Count', 
                     color='gender',
                     title="Does Age Matter? Win Distribution Based on Winner's Relative Age",
                     
                     # I need to change the colors of Male and Female so they are not confused with the Red and Blue Fight corners
                     color_discrete_map={
                         "FEMALE":"Pink",
                         "MALE":"Teal"
                     }
                     
                     ) 
    
    fig.update_traces(opacity=0.75)
   

    return fig




In [16]:
#sample output
# plot_age_difference_vs_winner('Both','All','All')

### Scatter Plot (Physical Attributes)
Now I want to see how see how fighters height and reach correlate to one another

To do this I will use a scatter plot

In [ ]:
#craete the function to create the dynamic chart
def plot_height_vs_reach(gender, weight_category, stance):
    #Standard filtering on the main dataset
    plot_df = subset_df.copy()
    
    if gender != 'Both':
        plot_df = plot_df[plot_df['gender'] == str.upper(gender)]

    if weight_category != 'All':
        plot_df = plot_df[plot_df['weight_class'] == weight_category]

    # Build the Unique Fighter Roster
    # Grab Red corner data and rename columns to be generic
    red_roster = plot_df[['R_fighter', 'R_Height_cms', 'R_Reach_cms', 'weight_class','R_Stance']].copy()
    red_roster = red_roster.rename(columns={
        'R_fighter': 'Fighter', 
        'R_Height_cms': 'Height_cms', 
        'R_Reach_cms': 'Reach_cms',
        'R_Stance':'Stance'
    })

    # Grab Blue corner data and rename columns to be generic
    blue_roster = plot_df[['B_fighter', 'B_Height_cms', 'B_Reach_cms', 'weight_class','B_Stance']].copy()
    blue_roster = blue_roster.rename(columns={
        'B_fighter': 'Fighter', 
        'B_Height_cms': 'Height_cms', 
        'B_Reach_cms': 'Reach_cms',
        'B_Stance':'Stance'
    })

    # Stack them together and drop duplicates based on the fighter's name
    # dropna() ensures we don't plot missing height/reach data
    unique_fighters = pd.concat([red_roster, blue_roster]).drop_duplicates(subset=['Fighter']).dropna()

    #apply stance filter
    if stance != 'All':
        unique_fighters = unique_fighters[unique_fighters['Stance'].str.strip()==stance]

    #Add Jitter to the UNIQUE fighters.  This will allow us to see it better when we plot the scatter plot
    unique_fighters['jittered_height'] = unique_fighters['Height_cms'] + np.random.uniform(-0.4, 0.4, len(unique_fighters))
    unique_fighters['jittered_reach'] = unique_fighters['Reach_cms'] + np.random.uniform(-0.4, 0.4, len(unique_fighters))

    #Build the scatter plot
    fig = px.scatter(
        unique_fighters, 
        x='jittered_height', 
        y='jittered_reach', 
        color='weight_class',
        hover_name='Fighter', # Now you can hover and see exactly who the outlier is
        title='Unique Fighter Height vs. Reach',
        opacity=0.6,
        
        # Clean up the hover tooltips
        hover_data={
            'jittered_height': False, 
            'jittered_reach': False,  
            'Height_cms': True,     
            'Reach_cms': True,
            'weight_class': True
        }
    )
    
    fig.update_layout(
        xaxis_title="Fighter Height (cm)",
        yaxis_title="Fighter Reach (cm)"
    )

    return fig
#sample output
# plot_height_vs_reach('Both','All','All')

### Line Chart (Performance Aging Curve)

Now I want to see how Performance of a fighter changes over their age

In [ ]:
#create selection tool that will be used to filter results by fighter performance metrics
performance_metric_toggle = pn.widgets.Select(
    name='fight_metric',
    options=['Avg_Strikes_Landed','Avg_TakeDown_Landed','Avg_Submission_Attmpt'],
    value='Avg_Strikes_Landed'
)


def plot_peak_age_curve(gender, weight_category,performance_metric,stance):
    #Filter by Gender
    plot_df = subset_df.copy()
    if gender == 'Female':
        plot_df = plot_df[plot_df['gender'] == 'FEMALE']
    elif gender == 'Male':
        plot_df = plot_df[plot_df['gender'] == 'MALE']

    #Extract all fighter appearances and determine if they won
    # Red Corner
    red_df = plot_df[['R_age', 'Winner', 'weight_class',
                      'R_avg_SIG_STR_landed','R_avg_TD_landed',
                      'R_avg_SUB_ATT','R_Stance']].copy()
    
    #rename red varaibles to have generic name
    red_df = red_df.rename(columns={'R_age': 'Age','R_avg_SIG_STR_landed': 'Avg_Strikes_Landed', 
                                    'R_avg_TD_landed': 'Avg_TakeDown_Landed','R_avg_SUB_ATT':'Avg_Submission_Attmpt',
                                    'R_Stance':'Stance'
                                    })
    red_df['Is_Winner'] = np.where(red_df['Winner'] == 'Red', 1, 0)

    # Blue Corner
    blue_df = plot_df[['B_age', 'Winner', 'weight_class',
                       'B_avg_SIG_STR_landed', 'B_avg_TD_landed',
                        'B_avg_SUB_ATT','B_Stance']].copy()
    
    #rename blue varaibles to have generic name
    blue_df = blue_df.rename(columns={'B_age': 'Age','B_avg_SIG_STR_landed': 'Avg_Strikes_Landed', 
                                      'B_avg_TD_landed': 'Avg_TakeDown_Landed','B_avg_SUB_ATT':'Avg_Submission_Attmpt',
                                      'B_Stance':'Stance'
                                      })
    blue_df['Is_Winner'] = np.where(blue_df['Winner'] == 'Blue', 1, 0)

    # Combine into a master list of all appearances and drop missing ages
    appearances = pd.concat([red_df, blue_df]).dropna(subset=['Age'])
    

    # Filter by stance
    if stance != 'All':
        appearances = appearances[appearances['Stance']==stance]

    #filter by weight class
    if weight_category != 'All':
        # Single line for the specific weight class selected
        appearances = appearances[appearances['weight_class'] == weight_category]
        color_col = None
        age_stats = appearances.groupby('Age').agg(Total_Wins=('Is_Winner', 'sum')).reset_index()
        title_text = f"Aging Curve: {performance_metric} in {weight_category} stance: {stance}"
 
    else:
        # Multiple lines if 'All' is selected
        color_col = 'weight_class'
        age_stats = appearances.groupby(['Age', 'weight_class']).agg(Total_Wins=('Is_Winner', 'sum')).reset_index()
        title_text = f"Aging Curve: {performance_metric} by Weight Class"
  
    #color code by weight class
    if color_col:
        age_stats = appearances.groupby(['Age', color_col]).agg(
            Total_Wins=('Is_Winner', 'sum'),
            Avg_Strikes_Landed=('Avg_Strikes_Landed', 'mean'),
            Avg_TakeDown_Landed=('Avg_TakeDown_Landed', 'mean'),
            Avg_Submission_Attmpt=('Avg_Submission_Attmpt', 'mean')
        ).reset_index()
    else:
        age_stats = appearances.groupby('Age').agg(
            Total_Wins=('Is_Winner', 'sum'),
            Avg_Strikes_Landed=('Avg_Strikes_Landed', 'mean'),
            Avg_TakeDown_Landed=('Avg_TakeDown_Landed', 'mean'),
            Avg_Submission_Attmpt=('Avg_Submission_Attmpt', 'mean')
        ).reset_index()

    #create the graph
    fig = px.line(
    age_stats, 
    x='Age', 
    y=performance_metric,
    color=color_col,
    markers=True, 
    title=title_text
    )
    #add titles 
    fig.update_layout(
        xaxis_title="Fighter Age",
        yaxis_title="Total Wins",
        hovermode="x" # Shows a clean tooltip for all lines at a specific age
    )
    
    # Smooth the lines slightly
    fig.update_traces(line_shape='spline')

    return fig



### Correlation Matrix (What correlates with a win and what does not?)

There are too many variables to analyze in the correlation matrix so we will create a function that allows you to select the top N varaibles with the highest and lowest correlation to our target

This will be a bar chart plot

In [ ]:


def plot_top_n_correlations(target_variable, gender_filter, weight_class_filter,stance, n=10):
    """
    Filters the dataframe, calculates dynamic correlations, and plots the top N.
    """
    #copy the subset_df so we do not override
    plot_df = subset_df.copy()
    
    #apply gender filter
    if gender_filter != 'Both':
        plot_df = plot_df[plot_df['gender'] == str.upper(gender_filter)]

    #apply weight class filter
    if weight_class_filter != 'All':
        plot_df = plot_df[plot_df['weight_class'] == weight_class_filter]
    
    # Apply stance filter
    if stance != 'All':
        plot_df = plot_df[plot_df['R_Stance']==stance]
        
    #create the target variable
    plot_df['Red_Is_Winner'] = np.where(plot_df['Winner'] == 'Red', 1, 0)
   
    numeric_df = plot_df.select_dtypes(include=['number', 'bool'])
    
    #calculate the correlation matrix dynamically on the filtered data
    correlation_matrix = numeric_df.corr()
    
    # Check if the target variable exists in the numeric subset to avoid errors
    if target_variable not in correlation_matrix.columns:
        return px.bar(title=f"Error: '{target_variable}' is not a numeric column.")

    #remove the target variable from its own column to avoid the 1.0 self-correlation
    target_corr = correlation_matrix[target_variable].drop(target_variable)
    
    #select the top N positive and negative correlations
    top_positive = target_corr.nlargest(n)
    top_negative = target_corr.nsmallest(n)
    
    #combine
    top_correlations = pd.concat([top_positive, top_negative])
    
    #create the bar chart
    fig = px.bar(
        top_correlations, 
        x=top_correlations.index, 
        y=top_correlations.values, 
        title=f'Top {n} Positive and Negative Correlations with {target_variable}',
        labels={'x': 'Variables', 'y': 'Correlation Coefficient'},
        color=top_correlations.values,
        color_continuous_scale='RdBu_r'
    )
    
    #angle the values on the xaxis to make them easier to read and fit on the screen
    fig.update_layout(xaxis_tickangle=-45)
    return fig


## The Dashboard

Putting it all together into a nice Panel Interactive Dashboard

In [23]:
pn.extension('plotly')

##### Correlation Chart #######

# Bind the filters to the function
interactive_chart = pn.bind(
    plot_top_n_correlations, 
    target_variable='Red_Is_Winner',
    gender_filter=gender_toggle,
    weight_class_filter=weigh_class_dropdown,
    stance=stance_dropdown,
    n=10
)
# Wrap it in a clean Plotly pane
correlation_chart_pane = pn.pane.Plotly(interactive_chart, height=600, sizing_mode="stretch_both")

#### Age and Weight Class Chart ####
#bind the toggle and dropdowns to the function
bound_age_chart = pn.bind(
    plot_age_difference_vs_winner,
    gender=gender_toggle,
    weight_category=weigh_class_dropdown,
    stance=stance_dropdown
)

#wrap it in a Plotly pane
age_chart_pane = pn.pane.Plotly(bound_age_chart, sizing_mode='stretch_width')

##### Height vs Reach Chart ########
#bind the filters to the function
bound_height_chart = pn.bind(
    plot_height_vs_reach,
    gender=gender_toggle,
    weight_category=weigh_class_dropdown,
    stance=stance_dropdown
)
#wrap it in a Plotly pane
height_chart_pane = pn.pane.Plotly(bound_height_chart, height=600, sizing_mode='stretch_width')


##### Wins over tiem ####
#bind the filters to the function
bound_win_time_chart = pn.bind(
    plot_peak_age_curve,
    gender=gender_toggle,
    weight_category=weigh_class_dropdown,
    performance_metric=performance_metric_toggle,
    stance=stance_dropdown
)
#wrap it in a Plotly pane
age_time_chart_pane = pn.pane.Plotly(bound_win_time_chart, sizing_mode='stretch_width')


##########Put it in a  Dashboard Template
template = pn.template.FastListTemplate(
    title='UFC Fight Predictor Dashboard',
    
    sidebar=[
        pn.pane.Markdown('### Gender & Weight Class Controls'),
        gender_toggle,
        weigh_class_dropdown,
        pn.layout.Divider(),
        pn.pane.Markdown('### Fighter Stance'),
        stance_dropdown,
        pn.layout.Divider(),
        pn.pane.Markdown('### Fighter Performance Metric'),
        performance_metric_toggle,


        ],      # Puts the dropdown on the left side
    
    main=[
        pn.Row(correlation_chart_pane,height_chart_pane),
        pn.Row(age_chart_pane,age_time_chart_pane)
        ],              # Puts the chart in the main center area in a 2X2 matrix
    
    theme='dark'                    # Or 'default' for a light theme
)

# Launch it!
template.show()

Launching server at http://localhost:64696
